In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.stats import qmc

### Function Description

You’re tasked with optimising a four-variable black-box function that represents the yield of a chemical process in a factory. The function is typically unimodal, with a single peak where yield is maximised. 

Your goal is to find the optimal combination of chemical inputs that delivers the highest possible yield, using systematic exploration and optimisation methods.

In [2]:
# Load data
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy').ravel()

In [3]:
df = pd.DataFrame(X, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y

print(df.describe())

              x1         x2         x3         x4       target
count  20.000000  20.000000  20.000000  20.000000    20.000000
mean    0.460387   0.498557   0.477944   0.494719   151.271876
std     0.227206   0.273619   0.246797   0.308458   251.955640
min     0.119879   0.038193   0.088947   0.072880     0.112940
25%     0.286214   0.286061   0.308915   0.180712    22.892661
50%     0.451188   0.535355   0.451095   0.458093    63.948431
75%     0.678976   0.740559   0.657973   0.792954   140.484809
max     0.836478   0.862540   0.879484   0.957644  1088.859618


In [4]:
# The "Smooth Mountain" Kernel
# RBF is mathematically perfect for unimodal functions. ARD is enabled.
initial_length_scales = np.array([1.0, 1.0, 1.0, 1.0])
kernel = RBF(length_scale=initial_length_scales, length_scale_bounds=(0.1, 10.0)) + \
         WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e2))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=True, 
    n_restarts_optimizer=25,
    random_state=42
)

model.fit(X, Y)

# High-Density 4D Search Grid
np.random.seed(42)
grid_size = 50000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 4))

mean, std = model.predict(search_grid, return_std=True)

# Beta Sweep
betas = [0.1, 0.5, 0.75, 1.0, 1.5, 2.0]
sweep_results = []

for b in betas:
    ucb_scores = mean + (b * std)
    best_idx = np.argmax(ucb_scores)
    best_point = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_point[0],
        'x2': best_point[1],
        'x3': best_point[2],
        'x4': best_point[3],
        'Pred Mean': mean[best_idx],
        'Uncertainty': std[best_idx]
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---
  Beta     x1     x2     x3     x4  Pred Mean  Uncertainty
0.1000 0.6046 0.9340 0.9660 0.9212  1192.3988      66.5502
0.5000 0.2213 0.9168 0.9742 0.9288  1189.8382      75.7536
0.7500 0.2213 0.9168 0.9742 0.9288  1189.8382      75.7536
1.0000 0.3561 0.9504 0.9977 0.9330  1178.8729      89.8441
1.5000 0.3547 0.9182 0.9966 0.9454  1167.1452      99.3896
2.0000 0.3093 0.7801 0.9932 0.9691  1130.5400     119.6753

--- Optimized Kernel Parameters ---
RBF(length_scale=[10, 4.57, 0.283, 0.177]) + WhiteKernel(noise_level=0.001)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 1 submission: x1 = 0.2213, x2 = 0.9168, x3 = 0.974, x4=0.9288

* The Problem Framing: 4-variable black-box function representing a chemical process. The target is the yield, which we want to maximize.
    * The function is unimodal. This means there are no "trick" local optima—just a single, smooth mountain with one true peak.
    * Current Peak: An exceptional yield of 1088.86, achieved by keeping Chemical 1 (x1​) low (~0.22) and maxing out the other three.

* The Surrogate Model: RBF + ARD + White Kernel.
    * RBF (Radial Basis Function): Since we know the landscape is unimodal. 
    * ARD (Automatic Relevance Determination): By assigning a length scale to each chemical, the model figures out which chemicals are highly sensitive and which ones don't matter.
    * White Kernel: The "Shock Absorber." Chemical yields naturally fluctuate. T

* Key Insights from the Model: The "Useless" Chemical: The ARD length scale for x1​ maxed out at 10.0. The model concluded that altering Chemical 1 barely impacts the yield, causing it to "drift" to random values (like 0.60) at low Beta.
    * The Highly Sensitive Chemicals: The length scales for x3​ and x4​ were tiny (0.283 and 0.177). These are the true drivers of the reaction.
    * Hyper-Confidence: The optimizer pushed the noise level to the lower bound (0.001), indicating it trusts the upward trend of the data implicitly.

* The Chosen Query (Beta = 0.5): [0.2213, 0.9168, 0.9742, 0.9288]

    The Logic: This was a textbook "surgical strike." By choosing Beta 0.5 instead of 0.1, we anchor the "stable" chemical (x1​) to our known successful baseline of ~0.22, preventing the model from making a lazy, unproven change. Meanwhile, it pushed the highly sensitive chemicals (x3​,x4​) even closer to their absolute limits to hunt for the true summit.

In [5]:
# ---------------------------------------------------
# WEEK1: NEW DATA HERE
# ---------------------------------------------------
w1_new_input = [0.2213, 0.9168, 0.9742, 0.9288]
w1_new_output = 2481.5089223693467

new_X = np.array(w1_new_input).reshape(1, -1)
new_Y = np.atleast_1d(w1_new_output)

X_updated_w1 = np.vstack((X, new_X))
Y_updated_w1 = np.append(Y, new_Y)

np.save('w1_updated_inputs.npy', X_updated_w1)
np.save('w1_updated_outputs.npy', Y_updated_w1)

df = pd.DataFrame(X_updated_w1, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w1

print(df.describe())
print(df.head(21))


              x1         x2         x3         x4       target
count  21.000000  21.000000  21.000000  21.000000    21.000000
mean    0.449002   0.518473   0.501575   0.515390   262.235545
std     0.227516   0.281875   0.263800   0.315216   564.693912
min     0.119879   0.038193   0.088947   0.072880     0.112940
25%     0.224189   0.316878   0.323806   0.190518    24.423088
50%     0.438933   0.536518   0.479592   0.473113    64.420147
75%     0.677491   0.774092   0.663893   0.814870   233.223610
max     0.836478   0.916800   0.974200   0.957644  2481.508922
          x1        x2        x3        x4       target
0   0.191447  0.038193  0.607418  0.414584    64.443440
1   0.758653  0.536518  0.656000  0.360342    18.301380
2   0.438350  0.804340  0.210245  0.151295     0.112940
3   0.706051  0.534192  0.264243  0.482088     4.210898
4   0.836478  0.193610  0.663893  0.785649   258.370525
5   0.683432  0.118663  0.829046  0.567577    78.434389
6   0.553621  0.667350  0.323806  0.81487

In [6]:
initial_length_scales = np.array([1.0, 1.0, 1.0, 1.0])
kernel = RBF(length_scale=initial_length_scales, length_scale_bounds=(0.1, 10.0)) + \
         WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e2))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=True, 
    n_restarts_optimizer=25,
    random_state=42
)

model.fit(X_updated_w1, Y_updated_w1)

np.random.seed(42)
grid_size = 50000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 4))

mean, std = model.predict(search_grid, return_std=True)

betas = [0.1, 0.5, 0.75, 1.0, 1.5, 2.0]
sweep_results = []

for b in betas:
    ucb_scores = mean + (b * std)
    best_idx = np.argmax(ucb_scores)
    best_point = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_point[0],
        'x2': best_point[1],
        'x3': best_point[2],
        'x4': best_point[3],
        'Pred Mean': mean[best_idx],
        'Uncertainty': std[best_idx]
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---
    Beta       x1       x2       x3       x4   Pred Mean  Uncertainty
0.100000 0.177193 0.731612 0.994130 0.981038 2522.998963    84.586617
0.500000 0.361593 0.858023 0.999172 0.995664 2513.449214   106.043239
0.750000 0.361593 0.858023 0.999172 0.995664 2513.449214   106.043239
1.000000 0.361593 0.858023 0.999172 0.995664 2513.449214   106.043239
1.500000 0.467106 0.107447 0.996744 0.999561 2507.809065   110.446106
2.000000 0.953098 0.206828 0.999375 0.986595 2499.586715   115.534678

--- Optimized Kernel Parameters ---
RBF(length_scale=[10, 10, 0.103, 0.617]) + WhiteKernel(noise_level=0.001)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 2 submission:

* Massive Jump: We saw a staggering increase in yield, leaping from the previous baseline to a record-breaking 2481.51.

* By choosing β=0.75, you are prioritizing "Exploitation". 

* The Consensus Signal: The fact that β=0.50 and β=0.75 both point to the exact same coordinates is a "lock-on" signal.

* Next Query Coordinate (β=0.75): [0.36159342, 0.85802259, 0.99917207, 0.99566390]


In [7]:
# ---------------------------------------------------
# WEEK2: NEW DATA HERE
# ---------------------------------------------------

w2_new_input = [0.361593, 0.858023, 0.999172, 0.995664]
w2_new_output = np.float64(2986.9379783299123)

w2_new_X = np.atleast_2d(w2_new_input)
w2_new_Y = np.atleast_1d(w2_new_output)

X_updated_w2 = np.vstack((X_updated_w1, w2_new_X))
Y_updated_w2 = np.append(Y_updated_w1, w2_new_Y)

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_updated_w2_scaled = scaler_x.fit_transform(X_updated_w2)
Y_updated_w2_scaled = scaler_y.fit_transform(Y_updated_w2.reshape(-1, 1))

df = pd.DataFrame(X_updated_w2, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w2

# print(df.describe())
print(df.head(22))

          x1        x2        x3        x4       target
0   0.191447  0.038193  0.607418  0.414584    64.443440
1   0.758653  0.536518  0.656000  0.360342    18.301380
2   0.438350  0.804340  0.210245  0.151295     0.112940
3   0.706051  0.534192  0.264243  0.482088     4.210898
4   0.836478  0.193610  0.663893  0.785649   258.370525
5   0.683432  0.118663  0.829046  0.567577    78.434389
6   0.553621  0.667350  0.323806  0.814870    57.571537
7   0.352356  0.322242  0.116979  0.473113   109.571876
8   0.153786  0.729382  0.422598  0.443074     8.847992
9   0.463442  0.630025  0.107906  0.957644   233.223610
10  0.677491  0.358510  0.479592  0.072880    24.423088
11  0.583973  0.147243  0.348097  0.428615    64.420147
12  0.306889  0.316878  0.622634  0.095399    63.476716
13  0.511142  0.817957  0.728710  0.112354    79.729130
14  0.438933  0.774092  0.378167  0.933696   355.806818
15  0.224189  0.846480  0.879484  0.878516  1088.859618
16  0.725262  0.479870  0.088947  0.759760    28

In [8]:
initial_length_scales = np.array([1.0, 1.0, 1.0, 1.0])
kernel = RBF(length_scale=initial_length_scales, length_scale_bounds=(0.1, 50.0)) + \
         WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-7, 1e2))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=25,
    random_state=42
)

model.fit(X_updated_w2_scaled, Y_updated_w2_scaled)

def acquisition_function(x_trial, beta, model):
    # L-BFGS-B passes the trial point in the scaled space
    x_trial = x_trial.reshape(1, -1)
    mean, std = model.predict(x_trial, return_std=True)
    ucb = mean[0] + (beta * std[0])
    return -ucb  # Minimize negative UCB to maximize UCB

betas = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
sweep_results = []

sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point 
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Precision Climbing with L-BFGS-B
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Translate predictions back to raw physical units
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---
    Beta       x1       x2       x3       x4   Pred Mean  Uncertainty
0.010000 0.776364 0.756612 1.000000 1.000000 3003.980161    37.017385
0.050000 0.887587 0.731196 1.000000 1.000000 3003.777701    43.609440
0.100000 1.000000 0.697235 1.000000 1.000000 3003.278846    50.730946
0.250000 1.000000 0.120757 1.000000 1.000000 3000.517485    64.178822
0.500000 1.000000 0.000000 1.000000 1.000000 2999.369528    68.573701
0.750000 1.000000 0.000000 1.000000 1.000000 2999.369528    68.573701
1.000000 1.000000 0.000000 1.000000 1.000000 2999.369528    68.573701

--- Optimized Kernel Parameters ---
RBF(length_scale=[50, 50, 0.696, 1.54]) + WhiteKernel(noise_level=0.000387)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 3 submission:

* Architectural Upgrade: Transitioned to the Hybrid Precision Search workflow. This combines a Sobol sequence for broad "scouting" with an L-BFGS-B local optimizer to navigate the continuous 4D space. 

* Data Standardization: Implemented explicit StandardScaler for both chemical inputs (X) and yield outputs (Y).

* After a successful phase at β=0.25, we dialed β=0.1. This represents a more aggressive move toward the predicted peak while still leaving room for minor adjustments.

* Next Query Coordinate (β=0.10): [1.0 , 0.697235 , 1.0 , 1.0] 

In [9]:
# ---------------------------------------------------
# WEEK3: NEW DATA HERE
# ---------------------------------------------------

w3_new_input = [1.000000, 0.697235, 1.000000, 1.000000]
w3_new_output = 5337.771630957252

w3_new_X = np.atleast_2d(w3_new_input)
w3_new_Y = np.atleast_1d(w3_new_output)

X_updated_w3 = np.vstack((X_updated_w2, w3_new_X))
Y_updated_w3 = np.append(Y_updated_w2, w3_new_Y)

X_updated_w3_scaled = scaler_x.fit_transform(X_updated_w3)
Y_updated_w3_scaled = scaler_y.fit_transform(Y_updated_w3.reshape(-1, 1))

df = pd.DataFrame(X_updated_w3, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w3

print(df.describe())

              x1         x2         x3         x4       target
count  23.000000  23.000000  23.000000  23.000000    23.000000
mean    0.469158   0.541008   0.544880   0.557341   601.376350
std     0.246536   0.279987   0.289569   0.331132  1295.398763
min     0.119879   0.038193   0.088947   0.072880     0.112940
25%     0.265539   0.319560   0.335952   0.275430    26.644920
50%     0.438933   0.630025   0.607418   0.482088    64.443440
75%     0.680462   0.789216   0.749436   0.864160   307.088672
max     1.000000   0.916800   1.000000   1.000000  5337.771631


In [10]:
initial_length_scales = np.array([1.0, 1.0, 1.0, 1.0])
kernel = RBF(length_scale=initial_length_scales, length_scale_bounds=(0.1, 100.0)) + \
         WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-7, 1e2))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=25,
    random_state=42
)

model.fit(X_updated_w3_scaled, Y_updated_w3_scaled)

betas = [0.005, 0.01, 0.05, 0.10, 0.25, 0.5]
sweep_results = []

for b in betas:
    # Find the best starting point 
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Precision Climbing with L-BFGS-B
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Translate predictions back to raw physical units
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---
    Beta       x1       x2       x3       x4   Pred Mean  Uncertainty
0.005000 1.000000 0.817364 1.000000 1.000000 5321.570418    77.440500
0.010000 1.000000 0.819721 1.000000 1.000000 5321.570360    77.448371
0.050000 1.000000 0.844677 1.000000 1.000000 5321.567600    77.541023
0.100000 1.000000 0.895310 1.000000 1.000000 5321.549953    77.781235
0.250000 1.000000 1.000000 1.000000 1.000000 5321.462277    78.497071
0.500000 1.000000 1.000000 1.000000 0.657148 5294.267164   137.282016

--- Optimized Kernel Parameters ---
RBF(length_scale=[3.68, 100, 0.369, 11.8]) + WhiteKernel(noise_level=0.00186)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 4 submission

Next point to query: [1.000000 , 0.819721 , 1.000000 , 1.000000] beta = 0.01 

* Extremely succesful last query 
* Moving even further in the same direction, going from beta = 0.10 to beta = 0.01

In [11]:
# ---------------------------------------------------
# WEEK4: NEW DATA HERE
# ---------------------------------------------------

w4_new_input = 	[1.000000, 0.819721, 1.000000, 1.000000]
w4_new_output = 6200.144699985609

w4_new_X = np.atleast_2d(w4_new_input)
w4_new_Y = np.atleast_1d(w4_new_output)

X_updated_w4 = np.vstack((X_updated_w3, w4_new_X))
Y_updated_w4 = np.append(Y_updated_w3, w4_new_Y)

X_updated_w4_scaled = scaler_x.fit_transform(X_updated_w4)
Y_updated_w4_scaled = scaler_y.fit_transform(Y_updated_w4.reshape(-1, 1))

df = pd.DataFrame(X_updated_w4, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w4

print(df.describe())

              x1         x2         x3         x4       target
count  24.000000  24.000000  24.000000  24.000000    24.000000
mean    0.491276   0.552621   0.563844   0.575785   834.658364
std     0.264346   0.279681   0.298052   0.336222  1706.221227
min     0.119879   0.038193   0.088947   0.072880     0.112940
25%     0.286214   0.320901   0.342025   0.317886    27.755836
50%     0.451188   0.634822   0.615026   0.524832    71.438914
75%     0.689087   0.807744   0.784883   0.891087   374.758303
max     1.000000   0.916800   1.000000   1.000000  6200.144700


In [12]:
initial_length_scales = np.array([1.0, 1.0, 1.0, 1.0])
kernel = RBF(length_scale=initial_length_scales, length_scale_bounds=(0.1, 100.0)) + \
         WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-7, 1e2))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=25,
    random_state=42
)

model.fit(X_updated_w4_scaled, Y_updated_w4_scaled)

betas = [0.005, 0.01, 0.05, 0.10, 0.25, 5]
sweep_results = []

for b in betas:
    # Find the best starting point 
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Precision Climbing with L-BFGS-B
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Translate predictions back to raw physical units
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---
    Beta       x1       x2       x3       x4   Pred Mean  Uncertainty
0.005000 1.000000 1.000000 1.000000 1.000000 6382.844808   346.984259
0.010000 1.000000 1.000000 1.000000 1.000000 6382.844808   346.984259
0.050000 1.000000 1.000000 1.000000 1.000000 6382.844808   346.984259
0.100000 1.000000 1.000000 1.000000 1.000000 6382.844808   346.984259
0.250000 1.000000 1.000000 1.000000 1.000000 6382.844808   346.984259
5.000000 1.000000 1.000000 1.000000 1.000000 6382.844808   346.984259

--- Optimized Kernel Parameters ---
RBF(length_scale=[2.74, 3.59, 0.803, 27.4]) + WhiteKernel(noise_level=0.0108)


### Week 5 submission:

Next point to query: [1.000000 1.000000 1.000000 1.000000] the full beta sweep suggests the same. 

* Possible bug / issue with the BO model
* Possible that pushing all x to the max is the solution, last query was very succesful

In [13]:
# ---------------------------------------------------
# WEEK5: NEW DATA HERE
# ---------------------------------------------------

w5_new_input = 	[1.000000, 1.000000, 1.000000, 1.000000]
w5_new_output = 8662.4825

w5_new_X = np.atleast_2d(w5_new_input)
w5_new_Y = np.atleast_1d(w5_new_output)

X_updated_w5 = np.vstack((X_updated_w4, w5_new_X))
Y_updated_w5 = np.append(Y_updated_w4, w5_new_Y)

X_updated_w5_scaled = scaler_x.fit_transform(X_updated_w5)
Y_updated_w5_scaled = scaler_y.fit_transform(Y_updated_w5.reshape(-1, 1))

df = pd.DataFrame(X_updated_w5, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w5

print(df.describe())

              x1         x2         x3         x4       target
count  25.000000  25.000000  25.000000  25.000000    25.000000
mean    0.511625   0.570516   0.581290   0.592754  1147.771330
std     0.278063   0.288042   0.304537   0.339902  2289.297814
min     0.119879   0.038193   0.088947   0.072880     0.112940
25%     0.306889   0.322242   0.348097   0.360342    28.866752
50%     0.463442   0.639619   0.622634   0.567577    78.434389
75%     0.706051   0.817957   0.829046   0.928800   431.612757
max     1.000000   1.000000   1.000000   1.000000  8662.482500


In [14]:
model.fit(X_updated_w5_scaled, Y_updated_w5_scaled)

betas = [0.01, 0.25, 1, 5, 10, 50]
sweep_results = []

for b in betas:
    # Find the best starting point 
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Precision Climbing with L-BFGS-B
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Translate predictions back to raw physical units
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 5: UNIMODAL YIELD OPTIMIZATION ---
     Beta       x1       x2       x3       x4   Pred Mean  Uncertainty
 0.010000 1.000000 1.000000 1.000000 1.000000 8371.107220   300.239130
 0.250000 1.000000 1.000000 1.000000 1.000000 8371.107220   300.239130
 1.000000 1.000000 1.000000 1.000000 1.000000 8371.107220   300.239130
 5.000000 1.000000 1.000000 1.000000 1.000000 8371.107220   300.239130
10.000000 1.000000 1.000000 0.000000 1.000000 1847.161275  1354.412592
50.000000 1.000000 1.000000 0.000000 1.000000 1847.161275  1354.412592

--- Optimized Kernel Parameters ---
RBF(length_scale=[2.94, 1.96, 2.29, 2.91]) + WhiteKernel(noise_level=0.00967)


### Week 6 submission

Next point to query: [1.000000 1.000000 0.000000 1.000000] beta = 10

* New maximum point where all the features are maxed - there's no further maximization possible 
* The function is described as unimodal, so there's a high chance this is the only peak 
* The next query is taking the extreme exploration approach (beta 10 - 50)
* Looking outside in case there's a peak outside the very prominent one that's been identified

In [15]:
# ---------------------------------------------------
# WEEK6: NEW DATA HERE
# ---------------------------------------------------

w6_new_input = 	[1.000000, 1.000000, 0.000000, 1.000000]
w6_new_output = 4440.5225

w6_new_X = np.atleast_2d(w6_new_input)
w6_new_Y = np.atleast_1d(w6_new_output)

X_updated_w6 = np.vstack((X_updated_w5, w6_new_X))
Y_updated_w6 = np.append(Y_updated_w5, w6_new_Y)

X_updated_w6_scaled = scaler_x.fit_transform(X_updated_w6)
Y_updated_w6_scaled = scaler_y.fit_transform(Y_updated_w6.reshape(-1, 1))

df = pd.DataFrame(X_updated_w6, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w6

print(df.describe())
# print(df.head(46))

              x1         x2         x3         x4       target
count  26.000000  26.000000  26.000000  26.000000    26.000000
mean    0.530409   0.587035   0.558933   0.608417  1274.415606
std     0.288790   0.294523   0.319420   0.342478  2334.150207
min     0.119879   0.038193   0.000000   0.072880     0.112940
25%     0.318256   0.331309   0.329879   0.373902    32.945456
50%     0.487292   0.653485   0.615026   0.663668    79.081759
75%     0.720459   0.819280   0.814325   0.932472   924.547903
max     1.000000   1.000000   1.000000   1.000000  8662.482500


### Week 7 submission

Next point to query [1.000000, 1.000000, 1.000000, 1.000000] 

Replication query, ask the same point and see if the same output is returned - to confirm the peak. 

In [16]:
# ---------------------------------------------------
# WEEK7: NEW DATA HERE
# ---------------------------------------------------

w7_new_input = 	[1.000000, 1.000000, 1.000000, 1.000000]
w7_new_output = 8662.4825

w7_new_X = np.atleast_2d(w7_new_input)
w7_new_Y = np.atleast_1d(w7_new_output)

X_updated_w7 = np.vstack((X_updated_w6, w7_new_X))
Y_updated_w7 = np.append(Y_updated_w6, w7_new_Y)

X_updated_w7_scaled = scaler_x.fit_transform(X_updated_w7)
Y_updated_w7_scaled = scaler_y.fit_transform(Y_updated_w7.reshape(-1, 1))

df = pd.DataFrame(X_updated_w7, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w7

# print(df.describe())
print(df.head(27))

          x1        x2        x3        x4       target
0   0.191447  0.038193  0.607418  0.414584    64.443440
1   0.758653  0.536518  0.656000  0.360342    18.301380
2   0.438350  0.804340  0.210245  0.151295     0.112940
3   0.706051  0.534192  0.264243  0.482088     4.210898
4   0.836478  0.193610  0.663893  0.785649   258.370525
5   0.683432  0.118663  0.829046  0.567577    78.434389
6   0.553621  0.667350  0.323806  0.814870    57.571537
7   0.352356  0.322242  0.116979  0.473113   109.571876
8   0.153786  0.729382  0.422598  0.443074     8.847992
9   0.463442  0.630025  0.107906  0.957644   233.223610
10  0.677491  0.358510  0.479592  0.072880    24.423088
11  0.583973  0.147243  0.348097  0.428615    64.420147
12  0.306889  0.316878  0.622634  0.095399    63.476716
13  0.511142  0.817957  0.728710  0.112354    79.729130
14  0.438933  0.774092  0.378167  0.933696   355.806818
15  0.224189  0.846480  0.879484  0.878516  1088.859618
16  0.725262  0.479870  0.088947  0.759760    28

### Week 8 submission

Next point to query [0.990000, 1.000000, 1.000000, 1.000000] 

* Peak confirmed
* Moving now to testing the sensitivities of each dimension, starting w/ x1

In [17]:
# ---------------------------------------------------
# WEEK8: NEW DATA HERE
# ---------------------------------------------------

w8_new_input = 	[0.990000, 1.000000, 1.000000, 1.000000]
w8_new_output = 8472.632447222944

w8_new_X = np.atleast_2d(w8_new_input)
w8_new_Y = np.atleast_1d(w8_new_output)

X_updated_w8 = np.vstack((X_updated_w7, w8_new_X))
Y_updated_w8 = np.append(Y_updated_w7, w8_new_Y)

X_updated_w8_scaled = scaler_x.fit_transform(X_updated_w8)
Y_updated_w8_scaled = scaler_y.fit_transform(Y_updated_w8.reshape(-1, 1))

df = pd.DataFrame(X_updated_w8, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w8

print(df.describe())

              x1         x2         x3         x4       target
count  28.000000  28.000000  28.000000  28.000000    28.000000
mean    0.563594   0.616533   0.590438   0.636387  1795.354310
std     0.303431   0.303395   0.328409   0.345181  2950.239966
min     0.119879   0.038193   0.000000   0.072880     0.112940
25%     0.340989   0.349443   0.342025   0.401023    41.102866
50%     0.532382   0.682292   0.632983   0.772705    94.650503
75%     0.778109   0.849366   0.903163   0.967149  2607.866186
max     1.000000   1.000000   1.000000   1.000000  8662.482500


### Week 9 submission

Next point to query [1.000000, 0.990000, 1.000000, 1.000000] 

* Peak confirmed
* Moving now to testing the sensitivities of each dimension, continuing w/ x2

In [18]:
# ---------------------------------------------------
# WEEK9: NEW DATA HERE
# ---------------------------------------------------

w9_new_input = [1.000000, 0.990000, 1.000000, 1.000000]
w9_new_output = 8472.632447222944

w9_new_X = np.atleast_2d(w9_new_input)
w9_new_Y = np.atleast_1d(w9_new_output)

X_updated_w9 = np.vstack((X_updated_w8, w9_new_X))
Y_updated_w9 = np.append(Y_updated_w8, w9_new_Y)

X_updated_w9_scaled = scaler_x.fit_transform(X_updated_w9)
Y_updated_w9_scaled = scaler_y.fit_transform(Y_updated_w9.reshape(-1, 1))

df = pd.DataFrame(X_updated_w9, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w9

# print(df.describe())
# print(df.head(39))

### Week 10 submission

Next point to query [1.000000 1.000000 0.990000 1.000000] 

* Exact same sensitivity between x1 and x2!
* Moving now to testing the sensitivities of each dimension, continuing w/ x3